In [1]:
pip install requests pandas openpyxl xlrd beautifulsoup4 lxml

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [openpyxl]2/3 [openpyxl]e]
Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
b3_index_composition.py
=======================
Extrai a composição (tickers + pesos) dos índices da B3 para um período selecionado.

Fontes utilizadas (em ordem de prioridade):
1. Arquivos históricos de carteiras teóricas da B3 (Excel)
2. Endpoint dinâmico do site da B3 (composição atual)
3. brapi.dev API (fallback para composição atual)

Índices suportados (exemplos):
  IBOV, IDIV, SMLL, IFIX, IGCT, MLCX, ITAG, IBRA, ICON, IEEX,
  IMAT, IMOB, INDX, ISEE, IFIX, UTIL

Instalação das dependências:
  pip install requests pandas openpyxl xlrd beautifulsoup4 lxml
"""

import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import BytesIO
from datetime import datetime, date
import time
import re
import os

# ---------------------------------------------------------------------------
# Configuração
# ---------------------------------------------------------------------------

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "pt-BR,pt;q=0.9",
}

# Mapeamento de índice → segmento de URL na B3
B3_INDEX_SLUGS = {
    "IBOV": "indices-amplos/indice-ibovespa-ibovespa",
    "IDIV": "indices-de-governanca/indice-de-dividendos-idiv",
    "SMLL": "indices-de-segmentos-e-setoriais/indice-small-cap-smll",
    "IFIX": "indices-de-segmentos-e-setoriais/indice-de-fundos-de-investimentos-imobiliarios-ifix",
    "IGCT": "indices-de-governanca/indice-de-governanca-corporativa-trade-igct",
    "ITAG": "indices-de-governanca/indice-de-acoes-com-tag-along-diferenciado-itag",
    "MLCX": "indices-amplos/indice-mid-large-cap-mlcx",
    "IBRA": "indices-amplos/indice-brasil-amplo-ibra",
    "ICON": "indices-de-segmentos-e-setoriais/indice-de-consumo-icon",
    "IEEX": "indices-de-segmentos-e-setoriais/indice-de-energia-eletrica-ieex",
    "IMAT": "indices-de-segmentos-e-setoriais/indice-de-materiais-basicos-imat",
    "INDX": "indices-de-segmentos-e-setoriais/indice-do-setor-industrial-indx",
    "UTIL": "indices-de-segmentos-e-setoriais/indice-de-utilidade-publica-util",
    "ISEE": "indices-de-sustentabilidade/indice-de-sustentabilidade-empresarial-ise",
}

# Rebalanceamentos da B3 (vigência das carteiras teóricas)
# Cada entrada é (ano, quadrimestre) → data de início da vigência
# Quadrimestres: 1=Jan-Abr, 2=Mai-Ago, 3=Set-Dez
QUADRIMESTRES = {
    (ano, quad): date(ano, [1, 5, 9][quad - 1], 1)
    for ano in range(2010, datetime.now().year + 1)
    for quad in [1, 2, 3]
}

# ---------------------------------------------------------------------------
# Utilitários
# ---------------------------------------------------------------------------

def _get_quadrimestre(target_date: date) -> tuple:
    """Retorna o (ano, quadrimestre) vigente para uma data."""
    if target_date.month <= 4:
        return (target_date.year, 1)
    elif target_date.month <= 8:
        return (target_date.year, 2)
    else:
        return (target_date.year, 3)


def _quadrimestres_no_periodo(start: date, end: date) -> list:
    """Lista todos os (ano, quad) que se sobrepõem ao período."""
    result = []
    for (ano, quad), inicio in sorted(QUADRIMESTRES.items()):
        fim_quad = QUADRIMESTRES.get((ano + (quad // 3), (quad % 3) + 1),
                                     date(ano + 1, 1, 1))
        if inicio <= end and fim_quad > start:
            result.append((ano, quad))
    return result

# ---------------------------------------------------------------------------
# Estratégia 1: Download direto do Excel da B3 (carteira teórica histórica)
# ---------------------------------------------------------------------------

def _build_b3_excel_url(index: str, ano: int, quad: int) -> str:
    """
    Monta a URL de download do Excel da carteira teórica.
    A B3 usa um padrão de URL com o código do índice e o período.
    """
    # Meses de início de cada quadrimestre
    mes = [1, 5, 9][quad - 1]
    # Formato usado pela B3 nos nomes de arquivo: MMAAAA
    periodo = f"{mes:02d}{ano}"

    # URL base para download de arquivos de índices
    base = "https://www.b3.com.br/lumis/portal/file/fileDownload.do"
    # Alternativa moderna (mais recente)
    base_new = (
        "https://sistemasdereferenciaexternos.b3.com.br/ords/fnet/published/"
        f"downloadFileFinancialInformation?idDocument="
    )

    # URL direta para carteira teórica (padrão observado no site da B3)
    url = (
        f"https://www.b3.com.br/data/files/C8/F3/08/E5/"
        f"296CE610A2CDD6A8AC094EA8/Indice_{index}_{periodo}.xlsx"
    )
    return url


def fetch_b3_excel(index: str, ano: int, quad: int,
                   session: requests.Session) -> pd.DataFrame | None:
    """
    Tenta baixar o Excel oficial da B3 para o quadrimestre especificado.
    Retorna DataFrame com colunas [ticker, participacao_pct, ano, quad] ou None.
    """
    mes = [1, 5, 9][quad - 1]
    periodo = f"{mes:02d}{ano}"

    # A B3 disponibiliza arquivos .xlsx e às vezes .xls
    urls_tentativas = [
        f"https://www.b3.com.br/data/files/C8/F3/08/E5/"
        f"296CE610A2CDD6A8AC094EA8/Indice_{index}_{periodo}.xlsx",
        f"https://www.b3.com.br/data/files/C8/F3/08/E5/"
        f"296CE610A2CDD6A8AC094EA8/Indice_{index}_{periodo}.xls",
        # URLs alternativas observadas para IBOV especificamente
        f"https://www.b3.com.br/data/files/C8/F3/08/E5/"
        f"296CE610A2CDD6A8AC094EA8/IBOVDia_{periodo}.xlsx",
    ]

    for url in urls_tentativas:
        try:
            r = session.get(url, headers=HEADERS, timeout=15)
            if r.status_code == 200 and len(r.content) > 1000:
                df = _parse_b3_excel(r.content, index, ano, quad)
                if df is not None and not df.empty:
                    print(f"  ✓ B3 Excel: {index} {ano}Q{quad} ({url.split('/')[-1]})")
                    return df
        except Exception as e:
            pass

    return None


def _parse_b3_excel(content: bytes, index: str,
                    ano: int, quad: int) -> pd.DataFrame | None:
    """
    Lê o Excel da B3 e extrai ticker + participação.
    O layout varia por índice, mas geralmente:
      - Linha de cabeçalho em torno da linha 7-10
      - Colunas: Código, Ação, Tipo, Qtde Teórica, Part.(%)
    """
    try:
        # Tenta ler como xlsx
        xl = pd.read_excel(BytesIO(content), sheet_name=0, header=None, engine="openpyxl")
    except Exception:
        try:
            xl = pd.read_excel(BytesIO(content), sheet_name=0, header=None, engine="xlrd")
        except Exception:
            return None

    # Procura a linha de cabeçalho (contém "Código" ou "Código" ou "Part.")
    header_row = None
    for i, row in xl.iterrows():
        row_str = " ".join(str(v).lower() for v in row.values)
        if "part" in row_str and ("c" in row_str or "ação" in row_str):
            header_row = i
            break

    if header_row is None:
        # Tenta heurística: primeira linha com mais de 3 valores não-nulos
        for i, row in xl.iterrows():
            if row.notna().sum() >= 3:
                header_row = i
                break

    if header_row is None:
        return None

    xl.columns = xl.iloc[header_row]
    df = xl.iloc[header_row + 1:].copy()
    df.columns = [str(c).strip() for c in df.columns]

    # Normaliza nomes de coluna
    col_map = {}
    for c in df.columns:
        cl = c.lower()
        if "código" in cl or "codigo" in cl or cl == "cod":
            col_map[c] = "ticker"
        elif "part" in cl and "%" in cl:
            col_map[c] = "participacao_pct"
        elif "part" in cl:
            col_map[c] = "participacao_pct"

    df = df.rename(columns=col_map)

    if "ticker" not in df.columns or "participacao_pct" not in df.columns:
        return None

    df = df[["ticker", "participacao_pct"]].copy()
    df = df.dropna(subset=["ticker", "participacao_pct"])
    df = df[df["ticker"].astype(str).str.match(r'^[A-Z]{4}\d{1,2}$')]
    df["participacao_pct"] = (
        pd.to_numeric(df["participacao_pct"], errors="coerce")
    )
    df = df.dropna(subset=["participacao_pct"])
    df["indice"] = index
    df["ano"] = ano
    df["quadrimestre"] = quad
    df["data_inicio_vigencia"] = date(ano, [1, 5, 9][quad - 1], 1)

    return df.reset_index(drop=True)

# ---------------------------------------------------------------------------
# Estratégia 2: Scraping do site da B3 (composição atual)
# ---------------------------------------------------------------------------

def fetch_b3_scraping(index: str, session: requests.Session) -> pd.DataFrame | None:
    """
    Faz scraping da página do índice na B3 para obter a composição atual.
    Funciona para qualquer índice listado em B3_INDEX_SLUGS.
    """
    slug = B3_INDEX_SLUGS.get(index.upper())
    if not slug:
        print(f"  ✗ Índice {index} não mapeado para scraping B3.")
        return None

    url = f"https://www.b3.com.br/pt_br/market-data-e-indices/indices/{slug}-composicao-da-carteira.htm"

    try:
        r = session.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
    except Exception as e:
        print(f"  ✗ Erro ao acessar B3 ({index}): {e}")
        return None

    soup = BeautifulSoup(r.text, "lxml")

    # Tenta encontrar tabela de composição
    tables = soup.find_all("table")
    for table in tables:
        df = _parse_html_table(table, index)
        if df is not None and not df.empty:
            print(f"  ✓ B3 Scraping: {index} (composição atual)")
            return df

    # Alternativa: dados podem estar em JSON embutido na página
    scripts = soup.find_all("script")
    for script in scripts:
        text = script.string or ""
        if "carteira" in text.lower() or "composicao" in text.lower():
            # Tenta extrair JSON
            match = re.search(r'\[(\{[^;]+)\]', text, re.DOTALL)
            if match:
                import json
                try:
                    data = json.loads("[" + match.group(1) + "]")
                    df = pd.DataFrame(data)
                    if not df.empty:
                        return _normalize_json_df(df, index)
                except Exception:
                    pass

    return None


def _parse_html_table(table, index: str) -> pd.DataFrame | None:
    """Extrai ticker e participação de uma tabela HTML."""
    try:
        dfs = pd.read_html(str(table), decimal=",", thousands=".")
        if not dfs:
            return None
        df = dfs[0]
    except Exception:
        return None

    # Normaliza colunas
    df.columns = [str(c).strip().lower() for c in df.columns]
    col_ticker = next((c for c in df.columns if "cód" in c or "cod" in c or "ticker" in c), None)
    col_part = next((c for c in df.columns if "part" in c), None)

    if col_ticker is None or col_part is None:
        return None

    df = df[[col_ticker, col_part]].copy()
    df.columns = ["ticker", "participacao_pct"]
    df = df.dropna()
    df = df[df["ticker"].astype(str).str.match(r'^[A-Z]{4}\d{1,2}$')]
    df["participacao_pct"] = pd.to_numeric(df["participacao_pct"], errors="coerce")
    df = df.dropna(subset=["participacao_pct"])

    today = date.today()
    ano, quad = _get_quadrimestre(today)
    df["indice"] = index
    df["ano"] = ano
    df["quadrimestre"] = quad
    df["data_inicio_vigencia"] = date(ano, [1, 5, 9][quad - 1], 1)
    return df.reset_index(drop=True)


def _normalize_json_df(df: pd.DataFrame, index: str) -> pd.DataFrame | None:
    """Normaliza DataFrame vindo de JSON embutido."""
    col_ticker = next((c for c in df.columns if "cod" in c.lower()), None)
    col_part = next((c for c in df.columns if "part" in c.lower()), None)
    if not col_ticker or not col_part:
        return None
    result = df[[col_ticker, col_part]].copy()
    result.columns = ["ticker", "participacao_pct"]
    today = date.today()
    ano, quad = _get_quadrimestre(today)
    result["indice"] = index
    result["ano"] = ano
    result["quadrimestre"] = quad
    result["data_inicio_vigencia"] = date(ano, [1, 5, 9][quad - 1], 1)
    return result.reset_index(drop=True)

# ---------------------------------------------------------------------------
# Estratégia 3: brapi.dev (fallback API gratuita)
# ---------------------------------------------------------------------------

def fetch_brapi(index: str, session: requests.Session,
                api_token: str = "") -> pd.DataFrame | None:
    """
    Usa a API brapi.dev para obter a composição atual do índice.
    Documentação: https://brapi.dev/docs
    """
    # brapi usa nomes em minúsculo para os índices
    url = f"https://brapi.dev/api/quote/list?search={index.lower()}&sortBy=close&sortOrder=desc"
    if api_token:
        url += f"&token={api_token}"

    try:
        r = session.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        print(f"  ✗ brapi.dev ({index}): {e}")
        return None

    stocks = data.get("stocks", [])
    if not stocks:
        return None

    rows = []
    for s in stocks:
        ticker = s.get("stock", "")
        if re.match(r'^[A-Z]{4}\d{1,2}$', ticker):
            rows.append({
                "ticker": ticker,
                "participacao_pct": None,  # brapi free não retorna pesos
                "nome": s.get("name", ""),
            })

    if not rows:
        return None

    df = pd.DataFrame(rows)
    today = date.today()
    ano, quad = _get_quadrimestre(today)
    df["indice"] = index
    df["ano"] = ano
    df["quadrimestre"] = quad
    df["data_inicio_vigencia"] = date(ano, [1, 5, 9][quad - 1], 1)
    print(f"  ✓ brapi.dev: {index} ({len(df)} tickers, sem pesos na versão gratuita)")
    return df.reset_index(drop=True)

# ---------------------------------------------------------------------------
# Função principal
# ---------------------------------------------------------------------------

def get_index_composition(
    indices: list[str],
    start_date: date | str,
    end_date: date | str | None = None,
    output_csv: str = "composicao_indices.csv",
    brapi_token: str = "",
    delay: float = 1.0,
) -> pd.DataFrame:
    """
    Retorna a composição dos índices selecionados para o período indicado.

    Parâmetros
    ----------
    indices      : lista de códigos de índice (ex: ["IBOV", "IDIV"])
    start_date   : data inicial do período (date ou "YYYY-MM-DD")
    end_date     : data final (padrão: hoje)
    output_csv   : nome do arquivo CSV de saída
    brapi_token  : token da brapi.dev (opcional, melhora rate limits)
    delay        : pausa entre requisições (segundos)

    Retorna
    -------
    DataFrame com colunas:
        indice, ticker, participacao_pct, ano, quadrimestre, data_inicio_vigencia
    """
    if isinstance(start_date, str):
        start_date = date.fromisoformat(start_date)
    if end_date is None:
        end_date = date.today()
    elif isinstance(end_date, str):
        end_date = date.fromisoformat(end_date)

    session = requests.Session()
    all_frames = []

    for index in [i.upper() for i in indices]:
        print(f"\n{'='*50}")
        print(f"Índice: {index} | Período: {start_date} → {end_date}")
        print(f"{'='*50}")

        quads = _quadrimestres_no_periodo(start_date, end_date)
        print(f"  Quadrimestres identificados: {quads}")

        found_any = False

        for (ano, quad) in quads:
            print(f"\n  → {ano} Q{quad} ({['Jan', 'Mai', 'Set'][quad-1]}-{['Abr', 'Ago', 'Dez'][quad-1]})")

            # Estratégia 1: Excel histórico da B3
            df = fetch_b3_excel(index, ano, quad, session)
            if df is not None:
                all_frames.append(df)
                found_any = True
                time.sleep(delay)
                continue

            # Estratégia 2: Scraping (só faz sentido para o quadrimestre atual)
            today_quad = _get_quadrimestre(date.today())
            if (ano, quad) == today_quad:
                df = fetch_b3_scraping(index, session)
                if df is not None:
                    all_frames.append(df)
                    found_any = True
                    time.sleep(delay)
                    continue

                # Estratégia 3: brapi.dev como último recurso
                df = fetch_brapi(index, session, brapi_token)
                if df is not None:
                    all_frames.append(df)
                    found_any = True

            time.sleep(delay)

        if not found_any:
            print(f"\n  ⚠ Não foi possível obter dados para {index}.")
            print("    Dica: Para dados históricos anteriores a 2018, use o")
            print("    Wayback Machine ou contate a B3 diretamente.")

    if not all_frames:
        print("\n⚠ Nenhum dado encontrado. Verifique sua conexão e os índices solicitados.")
        return pd.DataFrame()

    result = pd.concat(all_frames, ignore_index=True)

    # Ordena e exporta
    result = result.sort_values(["indice", "data_inicio_vigencia", "participacao_pct"],
                                ascending=[True, True, False])

    result.to_csv(output_csv, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV salvo: {output_csv}")
    print(f"   {len(result)} linhas | {result['ticker'].nunique()} tickers únicos")
    print(f"   Índices: {result['indice'].unique().tolist()}")

    return result


# ---------------------------------------------------------------------------
# Exemplo de uso
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    df = get_index_composition(
        indices=["IBOV", "IDIV"],
        start_date="2023-01-01",
        end_date="2024-12-31",
        output_csv="composicao_indices.csv",
        # brapi_token="SEU_TOKEN_AQUI",  # opcional
        delay=1.5,
    )

    if not df.empty:
        print("\n--- Prévia dos dados ---")
        print(df.head(10).to_string(index=False))

        print("\n--- Resumo por índice e quadrimestre ---")
        summary = df.groupby(["indice", "ano", "quadrimestre"]).agg(
            n_tickers=("ticker", "count"),
            peso_total=("participacao_pct", "sum"),
        )
        print(summary.to_string())


Índice: IBOV | Período: 2023-01-01 → 2024-12-31
  Quadrimestres identificados: [(2023, 1), (2023, 2), (2023, 3), (2024, 1), (2024, 2), (2024, 3)]

  → 2023 Q1 (Jan-Abr)

  → 2023 Q2 (Mai-Ago)

  → 2023 Q3 (Set-Dez)

  → 2024 Q1 (Jan-Abr)

  → 2024 Q2 (Mai-Ago)

  → 2024 Q3 (Set-Dez)

  ⚠ Não foi possível obter dados para IBOV.
    Dica: Para dados históricos anteriores a 2018, use o
    Wayback Machine ou contate a B3 diretamente.

Índice: IDIV | Período: 2023-01-01 → 2024-12-31
  Quadrimestres identificados: [(2023, 1), (2023, 2), (2023, 3), (2024, 1), (2024, 2), (2024, 3)]

  → 2023 Q1 (Jan-Abr)

  → 2023 Q2 (Mai-Ago)

  → 2023 Q3 (Set-Dez)

  → 2024 Q1 (Jan-Abr)

  → 2024 Q2 (Mai-Ago)

  → 2024 Q3 (Set-Dez)

  ⚠ Não foi possível obter dados para IDIV.
    Dica: Para dados históricos anteriores a 2018, use o
    Wayback Machine ou contate a B3 diretamente.

⚠ Nenhum dado encontrado. Verifique sua conexão e os índices solicitados.


In [3]:
"""
b3_composicao.py
================
Busca a composição atual dos índices da B3 diretamente do endpoint
interno que o site da B3 usa para renderizar as tabelas.

Retorna: CSV com colunas [indice, ticker, nome, tipo, qtde_teorica, participacao_pct]

Instalação:
    pip install requests pandas

Uso:
    python b3_composicao.py
"""

import requests
import pandas as pd
import time

# ---------------------------------------------------------------------------
# Endpoint interno da B3
# A B3 usa este endpoint Ajax para montar as tabelas de composição no site.
# Não é documentado publicamente, mas é estável há anos.
# ---------------------------------------------------------------------------
B3_ENDPOINT = "https://sistemasdereferenciaexternos.b3.com.br/ords/fnet/published/downloadFileFinancialInformation"

# Mapeamento índice → código interno da B3
# Para descobrir outros: inspecione as requisições de rede em
# https://www.b3.com.br/pt_br/market-data-e-indices/indices/
B3_INDEX_CODES = {
    "IBOV":  112,   # Ibovespa
    "IBRA":  121,   # Brasil Amplo
    "IDIV":  120,   # Dividendos
    "IGCT":  124,   # Gov. Corporativa Trade
    "IGCX":  123,   # Gov. Corporativa
    "ISEE":  119,   # Sustentabilidade
    "ITAG":  116,   # Tag Along
    "MLCX":  118,   # Mid-Large Cap
    "SMLL":  125,   # Small Cap
    "UTIL":  127,   # Utilidade Pública
    "ICON":  131,   # Consumo
    "IEEX":  129,   # Energia Elétrica
    "IMAT":  134,   # Materiais Básicos
    "IMOB":  135,   # Imobiliário (ações)
    "INDX":  130,   # Industrial
    "IFIX":  136,   # FIIs
}

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Referer": "https://www.b3.com.br/",
}


def fetch_index_composition(index_code: str, session: requests.Session) -> pd.DataFrame | None:
    """
    Busca a composição atual de um índice pelo endpoint da B3.

    O endpoint retorna um arquivo Excel (.xlsx) com a carteira teórica vigente.
    """
    index_code = index_code.upper()
    code_id = B3_INDEX_CODES.get(index_code)

    if code_id is None:
        print(f"  ✗ Índice '{index_code}' não encontrado. Disponíveis: {list(B3_INDEX_CODES.keys())}")
        return None

    params = {"idDocument": code_id}

    try:
        r = session.get(B3_ENDPOINT, params=params, headers=HEADERS, timeout=20)
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"  ✗ Erro ao acessar B3 para {index_code}: {e}")
        return None

    if len(r.content) < 500:
        print(f"  ✗ Resposta muito curta para {index_code} (provável erro da B3)")
        return None

    # Lê o Excel retornado
    try:
        # O Excel da B3 tem cabeçalhos nas primeiras linhas — pulamos com header=None
        xl = pd.read_excel(r.content, sheet_name=0, header=None, engine="openpyxl")
    except Exception as e:
        print(f"  ✗ Erro ao parsear Excel de {index_code}: {e}")
        return None

    return _parse_b3_excel(xl, index_code)


def _parse_b3_excel(xl: pd.DataFrame, index_code: str) -> pd.DataFrame | None:
    """
    Extrai os dados do Excel da B3.

    Layout típico das carteiras teóricas:
        Linhas 0-6: cabeçalho com título, data, etc.
        Linha 7: nomes das colunas
        Linhas 8+: dados dos ativos
        Última linha: totais

    Colunas habituais:
        Código | Ação | Tipo | Qtde. Teórica | Part. (%)
    """
    # Encontra a linha de cabeçalho (contém "Código" ou "Part")
    header_row = None
    for i, row in xl.iterrows():
        values = [str(v).strip().lower() for v in row if pd.notna(v)]
        if any("part" in v or "código" in v or "codigo" in v for v in values):
            header_row = i
            break

    if header_row is None:
        print(f"  ✗ Não encontrou cabeçalho no Excel de {index_code}")
        return None

    # Usa essa linha como cabeçalho
    df = xl.iloc[header_row + 1:].copy()
    df.columns = [str(xl.iloc[header_row, c]).strip() for c in range(xl.shape[1])]

    # Normaliza nomes das colunas
    rename = {}
    for col in df.columns:
        cl = col.lower()
        if "código" in cl or "codigo" in cl:
            rename[col] = "ticker"
        elif cl in ("ação", "acao", "nome"):
            rename[col] = "nome"
        elif "tipo" in cl:
            rename[col] = "tipo"
        elif "qtde" in cl or "quant" in cl:
            rename[col] = "qtde_teorica"
        elif "part" in cl:
            rename[col] = "participacao_pct"

    df = df.rename(columns=rename)

    # Mantém apenas colunas relevantes que existem
    cols = [c for c in ["ticker", "nome", "tipo", "qtde_teorica", "participacao_pct"] if c in df.columns]
    if "ticker" not in cols or "participacao_pct" not in cols:
        print(f"  ✗ Colunas essenciais não encontradas em {index_code}. Colunas: {list(df.columns)}")
        return None

    df = df[cols].copy()

    # Remove linhas sem ticker válido (rodapé com totais, linhas em branco)
    df = df[df["ticker"].astype(str).str.match(r'^[A-Z]{4}\d{1,2}$', na=False)]

    # Converte tipos
    df["participacao_pct"] = pd.to_numeric(df["participacao_pct"], errors="coerce")
    if "qtde_teorica" in df.columns:
        df["qtde_teorica"] = pd.to_numeric(
            df["qtde_teorica"].astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
            errors="coerce"
        )

    df = df.dropna(subset=["participacao_pct"])
    df.insert(0, "indice", index_code)

    return df.reset_index(drop=True)


def get_multiple_indices(
    indices: list[str],
    output_csv: str = "composicao_b3.csv",
    delay: float = 1.2,
) -> pd.DataFrame:
    """
    Busca a composição atual de múltiplos índices e salva em CSV.

    Parâmetros
    ----------
    indices    : lista de códigos (ex: ["IBOV", "IDIV", "SMLL"])
                 use ["ALL"] para buscar todos os índices mapeados
    output_csv : caminho do arquivo CSV de saída
    delay      : pausa entre requisições (segundos) — respeita o servidor da B3

    Retorna
    -------
    DataFrame com colunas: indice, ticker, nome, tipo, qtde_teorica, participacao_pct
    """
    if indices == ["ALL"]:
        indices = list(B3_INDEX_CODES.keys())

    session = requests.Session()
    frames = []

    for idx in indices:
        print(f"Buscando {idx}...", end=" ", flush=True)
        df = fetch_index_composition(idx, session)

        if df is not None and not df.empty:
            frames.append(df)
            print(f"✓ {len(df)} ativos")
        else:
            print("✗ sem dados")

        time.sleep(delay)

    if not frames:
        print("\nNenhum dado obtido. Verifique sua conexão.")
        return pd.DataFrame()

    result = pd.concat(frames, ignore_index=True)
    result = result.sort_values(["indice", "participacao_pct"], ascending=[True, False])

    result.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print(f"\n✅ Salvo: {output_csv}")
    print(f"   {len(result)} linhas | {result['indice'].nunique()} índices | {result['ticker'].nunique()} tickers únicos")

    return result


# ---------------------------------------------------------------------------
# Exemplo de uso
# ---------------------------------------------------------------------------
if __name__ == "__main__":

    # Opção A: índices específicos
    df = get_multiple_indices(
        indices=["IBOV", "IDIV", "SMLL", "IFIX"],
        output_csv="composicao_b3.csv",
    )

    # Opção B: todos os índices mapeados
    # df = get_multiple_indices(indices=["ALL"], output_csv="composicao_b3_todos.csv")

    if not df.empty:
        print("\n--- Prévia ---")
        print(df.head(10).to_string(index=False))

        print("\n--- Resumo por índice ---")
        print(
            df.groupby("indice")
            .agg(n_ativos=("ticker", "count"), peso_total=("participacao_pct", "sum"))
            .to_string()
        )

Buscando IBOV...   ✗ Erro ao acessar B3 para IBOV: HTTPSConnectionPool(host='sistemasdereferenciaexternos.b3.com.br', port=443): Max retries exceeded with url: /ords/fnet/published/downloadFileFinancialInformation?idDocument=112 (Caused by NameResolutionError("HTTPSConnection(host='sistemasdereferenciaexternos.b3.com.br', port=443): Failed to resolve 'sistemasdereferenciaexternos.b3.com.br' ([Errno -5] No address associated with hostname)"))
✗ sem dados
Buscando IDIV...   ✗ Erro ao acessar B3 para IDIV: HTTPSConnectionPool(host='sistemasdereferenciaexternos.b3.com.br', port=443): Max retries exceeded with url: /ords/fnet/published/downloadFileFinancialInformation?idDocument=120 (Caused by NameResolutionError("HTTPSConnection(host='sistemasdereferenciaexternos.b3.com.br', port=443): Failed to resolve 'sistemasdereferenciaexternos.b3.com.br' ([Errno -5] No address associated with hostname)"))
✗ sem dados
Buscando SMLL...   ✗ Erro ao acessar B3 para SMLL: HTTPSConnectionPool(host='sistema